In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import importlib

import model_utils
importlib.reload(model_utils)
from model_utils import *



In [ ]:
class SkipTFUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Intercept Keras/TensorFlow deserialization functions & model objects
        if "keras" in module or "tensorflow" in module or name in ["Functional", "Sequential", "Model"]:
            # Return a dummy placeholder class instead of recreating the Keras object
            return DummyKerasModel
        return super().find_class(module, name)

class DummyKerasModel:
    """Placeholder to safely swallow Keras unpickling calls without crashing."""
    def __init__(self, *args, **kwargs):
        pass
    
    @classmethod
    def _unpickle_model(cls, *args, **kwargs):
        return "<Skipped Keras Model>"

Models Results

In [ ]:
global_clients_metrics = []
costs_dict = {}
n_clients = [10, 15, 20]
iids = ['IID', 'Non-IID']
seeds = [42, 123, 456]
clients = n_clients[2]
iid = iids[1]
seed = seeds[0]

for clients in n_clients:
    for iid in iids:
            experiment = f'experiments/TON-n_{clients}-{iid}-seed_{seed}.pkl'
            with open(experiment, 'rb') as f:
                results = SkipTFUnpickler(f).load()
                rounds = results['telemetry']['rounds']
                global_loss = [round['global_loss'] for round in rounds]
                global_acc = [round['global_metrics']['accuracy'] for round in rounds]
                global_metrics = [round['global_metrics'] for round in rounds]
                client_metrics = [round['client_metrics'] for round in rounds]
                costs_dict[(clients, iid)] = results["telemetry"]['costs']
                loss_worst = []
                loss_best = []
                f1_worst = []
                f1_best = []
                acc_worst = []
                acc_best = []

                for i in range(len(client_metrics)):
                    _acc = [m['balanced_accuracy'] for m in client_metrics[i] if m['balanced_accuracy'] < 1.0]
                    _f1 = [m['f1_macro'] for m in client_metrics[i] if m['f1_macro'] < 1.0]
                    _loss = [m['loss'] for m in client_metrics[i] if m['loss'] < 1.0]
                    worst_index = np.argmax(_loss)
                    best_index = np.argmin(_loss)
                    loss_worst.append(_loss[worst_index])
                    f1_worst.append(_f1[worst_index])
                    acc_worst.append(_acc[worst_index])
                    loss_best.append(_loss[best_index])
                    f1_best.append(_f1[best_index])
                    acc_best.append(_acc[best_index])

                loss_dict = {(iid, clients): 
                             {'global_loss' : global_loss,
                              'global_acc' : global_acc,
                              'loss_worst' : loss_worst,
                              'loss_best' : loss_best,
                              'f1_worst' : f1_worst,
                              'f1_best' : f1_best,
                              'acc_worst' : acc_worst,
                              'acc_best' : acc_best}
                            }
                global_clients_metrics.append(loss_dict)


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 6))

for i in range(len(global_clients_metrics)):
    key = list(global_clients_metrics[i].keys())[0]
    distribution = key[0]
    clients = key[1]
    global_loss = global_clients_metrics[i][key]['global_loss']

    if distribution == 'IID':
        axs[0].plot(global_loss, label=f"{clients}")
    else:
        axs[1].plot(global_loss, label=f"{clients}")

axs[0].set_xlabel('Round')
axs[0].set_ylabel('Global Loss')
axs[0].set_title('Training Progress')
axs[0].legend()
axs[0].set_title('IID Distribution')
axs[1].set_title('Non-IID Distribution')
axs[1].legend()
axs[1].set_xlabel('Round')
axs[1].set_ylabel('Global Loss')
fig.suptitle('Training Loss Progress for Different Client Counts and Distributions')
plt.show()


In [ ]:
# 'loss_worst', 'loss_best', 'f1_worst', 'f1_best', 'acc_worst', 'acc_best'

fig, axs = plt.subplots(1, 2, figsize=(10, 6))

for i in range(len(global_clients_metrics)):
    key = list(global_clients_metrics[i].keys())[0]
    distribution = key[0]
    clients = key[1]

    if distribution == 'IID':
        m = sorted(global_clients_metrics[i][key]['acc_worst'])
        axs[0].plot(m, label=f"{clients}")
    else:
        m = sorted(global_clients_metrics[i][key]['acc_worst'])
        axs[1].plot(m, label=f"{clients}")

axs[0].set_xlabel('Round')
axs[0].set_ylabel('Loss')
axs[0].set_title('IID Distribution')
axs[0].legend()

axs[1].set_xlabel('Round')
axs[1].set_ylabel('Loss')
axs[1].set_title('Non-IID Distribution')
axs[1].legend()

fig.suptitle('Clients Worst Accuracy Progress for Different Client Counts and Distributions')
plt.show()


In [ ]:
# 'loss_worst', 'loss_best', 'f1_worst', 'f1_best', 'acc_worst', 'acc_best'

fig, axs = plt.subplots(1, 2, figsize=(10, 6))

for i in range(len(global_clients_metrics)):
    key = list(global_clients_metrics[i].keys())[0]
    distribution = key[0]
    clients = key[1]

    if distribution == 'IID':
        m = sorted(global_clients_metrics[i][key]['acc_best'])
        axs[0].plot(m, label=f"{clients}")
    else:
        m = sorted(global_clients_metrics[i][key]['acc_best'])
        axs[1].plot(m, label=f"{clients}")

axs[0].set_xlabel('Round')
axs[0].set_ylabel('Loss')
axs[0].set_title('IID Distribution')
axs[0].legend()

axs[1].set_xlabel('Round')
axs[1].set_ylabel('Loss')
axs[1].set_title('Non-IID Distribution')
axs[1].legend()

fig.suptitle('Clients Best Accuracy Progress for Different Client Counts and Distributions')
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 6))

threshold = 0.99
thr_idx = 0

colors = ['b', 'g', 'r']

for i in range(len(global_clients_metrics)):
    key = list(global_clients_metrics[i].keys())[0]
    distribution = key[0]
    clients = key[1]
    m = sorted(global_clients_metrics[i][key]['global_acc'])
    for idx, r in enumerate(m):
        if np.round(r, decimals=2) >= threshold:
            thr_idx = idx
            break

    if distribution == 'IID':
        axs[0].plot(m, color=colors[i % len(colors)], label=f"{clients}")
        axs[0].axvline(x=thr_idx, color=colors[i % len(colors)], linestyle='--', label=f'Round {thr_idx}')

    else:
        axs[1].plot(m, color=colors[i % len(colors)], label=f"{clients}")
        axs[1].axvline(x=thr_idx, color=colors[i % len(colors)], linestyle='--', label=f'Round {thr_idx}')

axs[0].set_xlabel('Round')
axs[0].set_ylabel('Accuracy')
axs[0].set_title('IID Distribution')
axs[0].legend()

axs[1].set_xlabel('Round')
axs[1].set_ylabel('Accuracy')
axs[1].set_title('Non-IID Distribution')
axs[1].legend()

fig.suptitle('Global Accuracy Progress for Different Client Counts and Distributions')
plt.show()

In [ ]:
costs_dict